# 1. SPARK STREAMING
- Môn học: Big Data - CO3137
- Ngày 14/04/2026
- Lớp: L01

| STT | Họ tên | MSSV |
| :---: | :--- | :---: |
| 1 | Lê Đình Đức | 2310774 |
| 2 | Nguyễn Văn Công Thành | 231xxx |

# 3. Exercise

### Exercise 1: Prepare movie data
- Read data from the following topics:
  - Movies ("Lab1_movies")
  - Ratings ("Lab1_ratings")
  - Tags ("Lab1_tags")
- Define schemas for movies, ratings, and tags.
- Convert the data to the defined schemas.

In [2]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

spark = (SparkSession.builder.appName("Lab1_Spark_Kafka")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.1.1")
    .config("spark.driver.memory", "4g")
    .master("local[*]")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [3]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

print("Ratings preview:")
df_ratings.show(3)
print("Movies preview:")
df_movies.show(3)
print("Tags preview:")
df_tags.show(3)


Ratings preview:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows
Movies preview:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
+-------+--------------------+--------------------+
only showing top 3 rows
Tags preview:
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
|     2|  60756|   will ferrell|1445714992|
+------+-------+---------------+-------

In [5]:
KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

# Delete existing topics and create them anew
admin_client = AdminClient({'bootstrap.servers': KAFKA_BROKERS})
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
print("Deleted existing topics (if any)")

new_topics = [
    NewTopic(topic="ratings", num_partitions=3, replication_factor=2),
    NewTopic(topic="movies", num_partitions=3, replication_factor=2),
    NewTopic(topic="tags", num_partitions=3, replication_factor=2)
]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result() # Wait for topic generation
        print(f"Topic '{topic}' created successfully.")
    except Exception as e:
        print(f"Failed to create topic '{topic}' (Might ignore if it's due to existing topic): {e}")


Deleted existing topics (if any)
Topic 'ratings' created successfully.
Topic 'movies' created successfully.
Topic 'tags' created successfully.


In [12]:
df_ratings.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "ratings").save()

df_movies.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "movies").save()

df_tags.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "tags").save()

print("Successfully written data to Kafka topics.")


Successfully written data to Kafka topics.


In [14]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

def read_and_parse_kafka(topic, schema, max_offsets=None):
    """Read from Kafka and parse JSON string"""
    reader = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest")

    if max_offsets:
        reader = reader.option("maxOffsetsPerTrigger", max_offsets)
        
    df_raw = reader.load()

    df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json_str") \
        .select(from_json(col("json_str"), schema).alias("data")) \
        .select("data.*")

    return df_parsed

movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Các dòng gọi hàm phía dưới giữ nguyên
df_movies_stream = read_and_parse_kafka("movies", movie_schema)
df_ratings_stream = read_and_parse_kafka("ratings", rating_schema, max_offsets=100)
df_tags_stream = read_and_parse_kafka("tags", tag_schema)

print("Movies DataFrame:")
df_movies_stream.printSchema()
print("Ratings DataFrame:")
df_ratings_stream.printSchema()
print("Tags DataFrame:")
df_tags_stream.printSchema()


Movies DataFrame:
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

Ratings DataFrame:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

Tags DataFrame:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)



In [15]:
import time

query_movies = df_movies_stream.writeStream \
    .format("json") \
    .option("path", "data/movies_static") \
    .option("checkpointLocation", "chk/movies") \
    .outputMode("append") \
    .start()

query_tags = df_tags_stream.writeStream \
    .format("json") \
    .option("path", "data/tags_static") \
    .option("checkpointLocation", "chk/tags") \
    .outputMode("append") \
    .start()

print("Waiting 10 seconds for stream processing...")
time.sleep(10)

query_movies.stop()
query_tags.stop()
print("\n Stream data saved to files.")

Waiting 10 seconds for stream processing...

 Stream data saved to files.


In [16]:
print("Reading movies from data/movies_static...")
df_movies = spark.read \
    .schema(movie_schema) \
    .json("data/movies_static")
print(f"Movies loaded: {df_movies.count()} rows")

print("Reading tags from data/tags_static...")
df_tags = spark.read \
    .schema(tag_schema) \
    .json("data/tags_static")
print(f"Tags loaded: {df_tags.count()} rows")

print("\n Static data successfully loaded from files.")

Reading movies from data/movies_static...
Movies loaded: 9742 rows
Reading tags from data/tags_static...
Tags loaded: 3683 rows

 Static data successfully loaded from files.


### Exercise 2: Hot genres
- Join ratings with movies to get genres.
- Write the output to the console every 5 seconds.

In [20]:
from pyspark.sql.functions import split, explode, desc 

df_genres = df_movies.select(
    "movieId", 
    "title", 
    explode(split(col("genres"), "\\|")).alias("genre")
)
print("Created genre mapping")


df_ratings_genres = df_ratings_stream.join(df_genres, on="movieId", how="inner")
print("Joined ratings with genres")


df_genre_agg = df_ratings_genres.groupBy("genre").count().orderBy(desc("count"))
print("Aggregated ratings by genre")


query = (
    df_genre_agg.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Starting stream output...")

query.awaitTermination(timeout=30)

query.stop() 
print("Complete")


Created genre mapping
Joined ratings with genres
Aggregated ratings by genre
Starting stream output...


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Drama    |47   |
|Comedy   |46   |
|Thriller |28   |
|Action   |24   |
|Romance  |23   |
|Adventure|18   |
|Crime    |15   |
|Mystery  |10   |
|War      |7    |
|Sci-Fi   |7    |
|Western  |6    |
|Horror   |5    |
|Fantasy  |4    |
|Children |2    |
|Musical  |1    |
|Animation|1    |
|IMAX     |1    |
+---------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Drama    |93   |
|Comedy   |76   |
|Thriller |52   |
|Adventure|47   |
|Action   |46   |
|Romance  |41   |
|Crime    |27   |
|Children |23   |
|Fantasy  |21   |
|Mystery  |17   |
|Sci-Fi   |16   |
|War      |14   |
|Musical  |14   |
|Animation|12   |
|Western  |9    |
|Horror   |7    |
|IMAX     |2    |
+---------+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Drama    |133  |
|Comedy   |107  |
|Action   |85   |
|Adventure|75   |
|Thriller |72   |
|Romance  |57   |
|Crime    |51   |
|Sci-Fi   |40   |
|Children |36   |
|Fantasy  |34   |
|Musical  |26   |
|War      |25   |
|Animation|22   |
|Mystery  |20   |
|Horror   |15   |
|Western  |13   |
|IMAX     |5    |
|Film-Noir|1    |
+---------+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Drama    |178  |
|Comedy   |131  |
|Action   |106  |
|Adventure|105  |
|Thriller |95   |
|Romance  |73   |
|Crime    |70   |
|Children |61   |
|Sci-Fi   |54   |
|Fantasy  |51   |
|Animation|40   |
|Mystery  |32   |
|War      |31   |
|Musical  |31   |
|Horror   |21   |
|Western  |14   |
|IMAX     |5    |
|Film-Noir|1    |
+---------+-----+



Complete


26/04/28 14:06:00 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/04/28 14:06:00 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.


### Exercise 3: Trending Now
- Use a 5-minute tumbling window on event time.
- Count by (window, movieId).
- Join with the title.
- Use dense rank over partition by window.
- Write the output to the console every 5 seconds.

In [27]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc, from_unixtime, window

print("Converting timestamp to TimestampType...")
windowed_counts = df_ratings_stream \
    .withColumn("timestamp", from_unixtime(col("timestamp")).cast("timestamp")) \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy(window("timestamp", "5 minutes"), "movieId") \
    .count()
print("Created 5-minute windows and counted by movieId")

print("Joining with movies...")
joined = windowed_counts.join(df_movies, on="movieId", how="inner")
print("Joined with movie titles")

def apply_ranking(batch_df, batch_id):
    window_spec = Window.partitionBy("window").orderBy(desc("count"), "movieId")
    result = batch_df.withColumn("rank", dense_rank().over(window_spec)) \
        .filter("rank <= 3") \
        .select("window", "movieId", "title", "count", "rank")
    
    print(f"\nBatch {batch_id}:")
    result.show(truncate=False)

query = joined.writeStream \
    .outputMode("complete") \
    .foreachBatch(apply_ranking) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Starting trending now stream...")
query.awaitTermination(timeout=30)
print("Complete")

Converting timestamp to TimestampType...
Created 5-minute windows and counted by movieId
Joining with movies...
Joined with movie titles
Starting trending now stream...

Batch 0:


+------------------------------------------+-------+----------------------------------+-----+----+
|window                                    |movieId|title                             |count|rank|
+------------------------------------------+-------+----------------------------------+-----+----+
|{1996-07-10 05:10:00, 1996-07-10 05:15:00}|10     |GoldenEye (1995)                  |1    |1   |
|{1996-07-10 05:10:00, 1996-07-10 05:15:00}|34     |Babe (1995)                       |1    |2   |
|{1996-07-10 05:10:00, 1996-07-10 05:15:00}|47     |Seven (a.k.a. Se7en) (1995)       |1    |3   |
|{1996-08-02 04:10:00, 1996-08-02 04:15:00}|193    |Showgirls (1995)                  |1    |1   |
|{1996-08-19 23:40:00, 1996-08-19 23:45:00}|10     |GoldenEye (1995)                  |1    |1   |
|{1996-08-19 23:40:00, 1996-08-19 23:45:00}|34     |Babe (1995)                       |1    |2   |
|{1996-08-19 23:40:00, 1996-08-19 23:45:00}|48     |Pocahontas (1995)                 |1    |3   |
|{1997-03-

### Exercise 4: Stream-Stream Join
- Perform a stream-stream join between tags and ratings.
- Use watermarks on both streams.
- Write the output to the console every 5 seconds.

In [29]:
print("Preparing tags stream with watermark...")
df_tags_with_watermark = df_tags_stream \
    .withColumn("timestamp", from_unixtime(col("timestamp")).cast("timestamp")) \
    .withColumnRenamed("timestamp", "tagTime") \
    .withWatermark("tagTime", "10 minutes")

print("Preparing ratings stream with watermark...")
df_ratings_with_watermark = df_ratings_stream \
    .withColumn("timestamp", from_unixtime(col("timestamp")).cast("timestamp")) \
    .withColumnRenamed("timestamp", "ratingTime") \
    .withWatermark("ratingTime", "10 minutes")

print("Performing stream-stream join on movieId...")
stream_stream_join = df_tags_with_watermark.join(
    df_ratings_with_watermark,
    on="movieId",
    how="inner"
).select("movieId", "tag", "rating", "ratingTime", "tagTime")

query = (
    stream_stream_join.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Starting stream-stream join...")
query.awaitTermination(timeout=30)
print("Complete")

Preparing tags stream with watermark...
Preparing ratings stream with watermark...
Performing stream-stream join on movieId...
Starting stream-stream join...


-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-------+------+-------------------+-------------------+
|movieId|tag    |rating|ratingTime         |tagTime            |
+-------+-------+------+-------------------+-------------------+
|1580   |aliens |3.0   |2000-07-31 01:18:45|2006-01-14 09:25:19|
|1088   |dance  |3.0   |2016-02-16 17:41:15|2006-01-27 03:20:56|
|1088   |music  |3.0   |2016-02-16 17:41:15|2006-01-27 03:20:56|
|1580   |aliens |3.5   |2016-02-18 05:33:55|2006-01-14 09:25:19|
|1580   |aliens |4.5   |2014-08-10 03:57:13|2006-01-14 09:25:19|
|1580   |aliens |3.0   |2000-07-04 11:59:18|2006-01-14 09:25:19|
|1580   |aliens |3.0   |2009-02-13 16:08:37|2006-01-14 09:25:19|
|1645   |lawyers|2.5   |2009-05-11 16:12:31|2006-01-16 08:14:55|
|1580   |aliens |2.5   |2017-12-26 04:39:55|2006-01-14 09:25:19|
|1088   |dance  |4.0   |2009-01-03 03:55:36|2006-01-27 03:20:56|
|1088   |music  |4.0   |2009-01-03 03:55:36|2006-01-27 03: